# Plot Constructor Example

This notebook demonstrates how to build stacked plots from processor results.

## Set up environment

In [ ]:
# --- Notebook bootstrap: works locally + in Colab ---
from pathlib import Path
import os, sys

def _find_project_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / "pyproject.toml").exists() and (parent / "app").exists():
            return parent
    return p

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

# If running in Colab and repo files aren't present, clone automatically
if IN_COLAB and not (Path.cwd() / "app").exists():
    import subprocess
    if not Path("Polar-lights").exists():
        subprocess.check_call([
            "git", "clone", "-b", "main",
            "https://github.com/Yuri-ga1/Polar-lights.git"
        ])
    os.chdir("Polar-lights")

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Install deps in Colab (safe to re-run)
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "poetry"])
    subprocess.check_call(["poetry", "config", "virtualenvs.create", "false"])
    subprocess.check_call(["poetry", "install", "--no-interaction", "--no-ansi"])

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)

In [ ]:
from app.visualization.plot_settings import set_plt_def_params
from datetime import datetime, timedelta

set_plt_def_params()

## 1) User parameters

Set everything in one place: date, optional station codes, SIMURG email and requested plots.

In [ ]:
DATE_STR = "2025-11-12"
BASE_DIR = "files"

# Optional data-source parameters (global defaults)
IONOSONDE_CODE = None
COSMIC_STATIONS = ["OULU", "APTY"]
SIMURG_EMAIL = "Storm_Plotter_Jupyter_Notebook@gmail.com"

# Requested plots for PlotConstructor (order matters)
PLOTS_TO_DRAW = [
    "ROTI",
    "GIM",
    "Dst",
    "Kp",
]

# Per-plot PIPELINE params (not matplotlib style params)
PLOT_SPECS = [
    {
        "name": "ROTI",
        "params": {
            "email": "Storm_Plotter_Jupyter_Notebook@gmail.com",
            "plot_times": [datetime(2025, 11, 12, 2, 0, 0), datetime(2025, 11, 12, 4, 0, 0)],
            "time": datetime(2025, 11, 12, 2, 0, 0),
        },
    },
    {
        "name": "GIM",
        "params": {
            "product_type": "uqrg",
            "time": datetime(2025, 11, 12, 0, 0, 0),
        },
    },
    {"name": "Dst"},
    {"name": "Kp"},
]


## 2) Data loader/orchestrator (decomposed)

This helper inspects requested plot names, downloads only required sources, processes them with corresponding Processor classes and returns `processor_results`.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path
from dataclasses import dataclass
from typing import Any
from datetime import datetime, timedelta

import pandas as pd

from app.pipeline.observation_pipeline import (
    collect_observation_links,
    load_observations_from_csv,
    parse_and_save_observations,
)
from app.storage.hdf5_storage import ObservationHDF5Storage

from app.gfz.gfz_downloader import GfzDownloader
from app.gfz.gfz_processor import GfzProcessor
from app.kyoto.kyoto_dst_downloader import KyotoDstDownloader
from app.kyoto.kyoto_dst_processor import KyotoProcessor
from app.simurg.gim_downloader import GimDownloader
from app.simurg.gim_processor import GimProcessor
from app.simurg.simurg_client import SimurgClient
from app.simurg.simurg_downloader import RotiDownloader, AdjustedTecDownloader
from app.simurg.simurg_processor import SimurgProcessor, DataProduct


@dataclass
class ConstructorDataConfig:
    date_str: str
    base_dir: str = "files"
    simurg_email: str | None = None


class PlotConstructorDataLoader:
    def __init__(self, config: ConstructorDataConfig) -> None:
        self.config = config

        parents_dir = Path.cwd().parent
        download_dir = parents_dir / config.base_dir
        self.date_dir = os.path.join(download_dir, config.date_str)

    @staticmethod
    def _normalize(name: str) -> str:
        return " ".join(name.lower().replace("_", " ").split())

    def _contains(self, requested: set[str], *names: str) -> bool:
        return any(self._normalize(name) in requested for name in names)

    @staticmethod
    def _merge_plot_params(plots: list[str | dict[str, Any]]) -> dict[str, dict[str, Any]]:
        merged: dict[str, dict[str, Any]] = {}
        for item in plots:
            if isinstance(item, str):
                continue
            name = " ".join(str(item.get("name", "")).lower().replace("_", " ").split())
            if not name:
                continue
            merged.setdefault(name, {}).update(dict(item.get("params", {})))
        return merged

    def _safe_download(self, fn):
        try:
            return fn()
        except Exception as exc:
            print(f"Download warning: {exc}")
            return None

    def _load_kp(self):
        out_dir = os.path.join(self.date_dir, "kp")
        os.makedirs(out_dir, exist_ok=True)
        self._safe_download(lambda: GfzDownloader(out_dir=out_dir).download(date_str=self.config.date_str, fmt="kp2"))
        return GfzProcessor(folder_path=out_dir).load(date_str=self.config.date_str)

    def _load_dst(self):
        out_dir = os.path.join(self.date_dir, "kyoto")
        os.makedirs(out_dir, exist_ok=True)
        self._safe_download(lambda: KyotoDstDownloader(out_dir=out_dir).download(self.config.date_str))
        return KyotoProcessor(folder_path=out_dir).load(self.config.date_str)

    def _simurg_client(self, email: str | None = None) -> SimurgClient | None:
        resolved_email = email or self.config.simurg_email
        if not resolved_email:
            return None
        return SimurgClient(email=resolved_email)

    def _load_roti(self, params: dict[str, Any] | None = None):
        params = params or {}
        client = self._simurg_client(params.get("email"))
        if client is None:
            print("SIMURG email is missing, skip ROTI download")
            return None

        out_dir = os.path.join(self.date_dir, "simurg")
        os.makedirs(out_dir, exist_ok=True)
        self._safe_download(lambda: RotiDownloader(client=client, out_dir=out_dir).download(self.config.date_str))

        target_date = datetime.strptime(self.config.date_str, "%Y-%m-%d").date() - timedelta(days=1)
        return SimurgProcessor(folder_path=out_dir).load(target_date, product_type=DataProduct.ROTI)

    def _load_adjusted_tec(self, params: dict[str, Any] | None = None):
        params = params or {}
        client = self._simurg_client(params.get("email"))
        if client is None:
            print("SIMURG email is missing, skip adjusted TEC download")
            return None

        out_dir = os.path.join(self.date_dir, "simurg")
        os.makedirs(out_dir, exist_ok=True)
        self._safe_download(lambda: AdjustedTecDownloader(client=client, out_dir=out_dir).download(self.config.date_str))
        return SimurgProcessor(folder_path=out_dir).load(self.config.date_str, product_type=DataProduct.TEC_ADJUSTED)

    def _load_gim(self, params: dict[str, Any] | None = None):
        params = params or {}
        product_type = str(params.get("product_type", "uqrg")).lower()
        out_dir = os.path.join(self.date_dir, "gim")
        os.makedirs(out_dir, exist_ok=True)
        self._safe_download(lambda: GimDownloader(out_dir=out_dir, gim_type=product_type).download(self.config.date_str))
        return GimProcessor(folder_path=out_dir).load(self.config.date_str)

    def _aurora_stub(self, date_str: str) -> pd.DataFrame:
        download_dir = self.date_dir
        os.makedirs(download_dir, exist_ok=True)

        h5_path = os.path.join(download_dir, "spaceweather_observations.h5")
        csv_path = os.path.join(download_dir, "aurora_data.csv")

        observations: list[dict[str, str]] = []
        storage = ObservationHDF5Storage(h5_path)

        date_iso = date_str
        date_slash = date_str.replace("-", "/")

        cached_rows = load_observations_from_csv(csv_path, date_iso)
        if cached_rows:
            observations.extend(cached_rows)

        if storage.has_date(date_iso):
            observations.extend(
                parse_and_save_observations(
                    h5_path,
                    csv_path,
                    dates=[date_iso],
                )
            )

        collect_observation_links(date_slash, h5_path)
        observations.extend(
            parse_and_save_observations(
                h5_path,
                csv_path,
                dates=[date_iso],
            )
        )

        if not observations:
            return pd.DataFrame(columns=["date", "time", "lat", "lon", "colors"])

        return pd.DataFrame(observations)

    def load_for_requested_plots(self, plots: list[str | dict[str, Any]]) -> dict[str, Any]:
        names = {
            self._normalize(item if isinstance(item, str) else item.get("name", ""))
            for item in plots
        }
        params_by_plot = self._merge_plot_params(plots)

        results: dict[str, Any] = {}

        if self._contains(names, "kp"):
            results["Kp"] = self._load_kp()
        if self._contains(names, "dst"):
            results["Dst"] = self._load_dst()
        if self._contains(names, "roti", "keogram"):
            results["ROTI"] = self._load_roti(params_by_plot.get("roti"))
        if self._contains(names, "adjusted tec", "tec adjusted"):
            results["Adjusted TEC"] = self._load_adjusted_tec(params_by_plot.get("adjusted tec") or params_by_plot.get("tec adjusted"))
        if self._contains(names, "gim"):
            results["GIM"] = self._load_gim(params_by_plot.get("gim"))
        if self._contains(names, "aurora observation", "aurora"):
            results["Aurora Observation"] = self._aurora_stub(self.config.date_str)

        return results


## 3) Build processor results for constructor

In [ ]:
from app.visualization import PlotConstructor

loader = PlotConstructorDataLoader(
    ConstructorDataConfig(
        date_str=DATE_STR,
        base_dir=BASE_DIR,
        simurg_email=SIMURG_EMAIL,
    )
)

processor_results = loader.load_for_requested_plots(PLOT_SPECS)
plotter = PlotConstructor(processor_results)
plotter.available_plots()

## 4) Plot stacked charts (same order as input list)

In [ ]:
plotter.plot(PLOT_SPECS);